# Trust Layer — Uzbek Car Recognizer (6-class, open-set)

Turns the trained model (v2: sub-center ArcFace @384, **0.950 acc / 0.936 macro-F1** on the
sealed test) into a system that **knows when it doesn't know** — so it can hold a very high
**precision on the five known models** by *abstaining* on the hard cases.

Steps: **temperature-scale** the confidences → find the threshold for **≥99% precision on the
five known classes** on validation → apply it **once** to the sealed test → precision–coverage
curve → reliability diagrams → a grid of the most confident precision-breaking mistakes.

**Open-set framing:** `others` is a *reject* label, not a business answer. We "answer" only when
the model predicts one of the five known models **and** is confident enough; predicting `others`
or falling below the threshold both mean *route to a human / don't count*. Precision is measured
on the answers that matter.

**Prerequisite:** run the Model Gate first, so the dataset zip (`dataset_final.zip`) **and** the
artifacts zip (`modelgate_v2_artifacts.zip`) are both in Drive `CapstoneCars/`. Set
**Runtime → T4 GPU**.

In [ ]:
# ── Cell 1 · Setup + model definitions (must match the Model Gate) ──
!pip install -q timm
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, json, glob
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import timm
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", dev)

# These classes are COPIED VERBATIM from model_gate_v2 so the saved weights reload exactly.
class SubCenterArcFace(nn.Module):
    def __init__(self, in_features, num_classes, K=3, s=30.0, m=0.30):
        super().__init__()
        self.num_classes, self.K, self.s, self.m = num_classes, K, s, m
        self.W = nn.Parameter(torch.empty(num_classes*K, in_features))
        nn.init.xavier_uniform_(self.W)
    def forward(self, feat, labels=None):
        f = F.normalize(feat, dim=1)
        w = F.normalize(self.W, dim=1)
        cos = (f @ w.t()).view(-1, self.num_classes, self.K).amax(dim=2)
        cos = cos.float().clamp(-1+1e-6, 1-1e-6)
        if labels is None:
            return self.s * cos
        theta  = torch.acos(cos)
        margin = torch.zeros_like(cos).scatter_(1, labels.view(-1,1), self.m)
        return self.s * torch.cos(theta + margin)

class ArcModel(nn.Module):
    def __init__(self, model_id, num_classes, K=3, s=30.0, m=0.30, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_id, pretrained=pretrained,
                                          num_classes=0, global_pool='avg')
        self.head = SubCenterArcFace(self.backbone.num_features, num_classes, K, s, m)
    def forward(self, x, labels=None):
        return self.head(self.backbone(x), labels)

class SnapMixNet(nn.Module):
    def __init__(self, model_id, num_classes, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_id, pretrained=pretrained,
                                          num_classes=0, global_pool='')
        self.C  = self.backbone.num_features
        self.fc = nn.Linear(self.C, num_classes)
    def forward(self, x):
        f = self.backbone.forward_features(x)
        return self.fc(F.adaptive_avg_pool2d(f, 1).flatten(1))
print("model classes ready")

In [ ]:
# ── Cell 2 · Reload the trained model + the split from Drive ────
from google.colab import drive
drive.mount('/content/drive')
ds = (glob.glob('/content/drive/MyDrive/**/dataset_final.zip', recursive=True)
      or glob.glob('/content/drive/MyDrive/**/dataset_split.zip', recursive=True))[0]
arts = (glob.glob('/content/drive/MyDrive/**/modelgate_v2_artifacts.zip', recursive=True)
        or glob.glob('/content/drive/MyDrive/**/modelgate_artifacts.zip', recursive=True))
ART_ZIP = arts[0]
print("data:", ds); print("artifacts:", ART_ZIP)
!rm -rf /content/dataset /content/artifacts
!unzip -o -q "$ds" -d /content
!unzip -o -q "$ART_ZIP" -d /content

cfg = json.load(open('/content/artifacts/config.json'))
CLASSES = cfg['classes']
print("winner:", cfg['winner'], "| head:", cfg.get('head','linear'), "| classes:", CLASSES)

def build_model(cfg):
    h = cfg.get('head', 'linear')
    if   h == 'arcface': m = ArcModel(cfg['model_id'], len(CLASSES), pretrained=False, **cfg['arc'])
    elif h == 'snapmix': m = SnapMixNet(cfg['model_id'], len(CLASSES), pretrained=False)
    else:                m = timm.create_model(cfg['model_id'], pretrained=False, num_classes=len(CLASSES))
    m.load_state_dict(torch.load('/content/artifacts/model.pt', map_location=dev))
    return m.to(dev).eval()

model = build_model(cfg)
print("model reloaded ✓ (head =", cfg.get('head','linear'), ")")

In [ ]:
# ── Cell 3 · Val/test loaders + raw logits ──────────────────────
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
IMG = cfg['img_size']; MEAN, STD = tuple(cfg['mean']), tuple(cfg['std'])
RESIZE = int(round(IMG / 0.875))                       # 384 -> 438 (matches the Model Gate eval_tf)
eval_tf = transforms.Compose([transforms.Resize(RESIZE), transforms.CenterCrop(IMG),
                              transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
val_ds  = datasets.ImageFolder('/content/dataset/val',  eval_tf)
test_ds = datasets.ImageFolder('/content/dataset/test', eval_tf)
assert val_ds.classes == CLASSES, "class order mismatch!"
val_loader  = DataLoader(val_ds,  48, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, 48, shuffle=False, num_workers=2)

@torch.no_grad()
def get_logits(loader):
    L, Y = [], []
    for x,y in loader:
        L.append(model(x.to(dev)).float().cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)
val_logits, val_y = get_logits(val_loader)
test_logits, test_y = get_logits(test_loader)
print("val", tuple(val_logits.shape), "| test", tuple(test_logits.shape))

In [ ]:
# ── Cell 4 · Temperature scaling (calibrate the confidences) ────
# Fit ONE scalar T on validation to make the softmax probabilities honest (Guo et al. 2017).
# ArcFace logits are s*cos (peaked), so expect T > 1 to soften them.
T = torch.nn.Parameter(torch.ones(1))
opt = torch.optim.LBFGS([T], lr=0.05, max_iter=200)
def closure():
    opt.zero_grad(); loss = F.cross_entropy(val_logits/T, val_y); loss.backward(); return loss
opt.step(closure)
T = max(0.05, float(T.detach()))
print("fitted temperature T =", round(T, 3))

def ece(logits, y, T=1.0, bins=15):
    p = torch.softmax(logits/T, 1); conf, pred = p.max(1)
    conf, pred, y = conf.numpy(), pred.numpy(), y.numpy()
    e, edges = 0.0, np.linspace(0, 1, bins+1)
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i+1])
        if m.sum(): e += abs((pred[m]==y[m]).mean() - conf[m].mean()) * m.mean()
    return e
print(f"ECE (test) — before {ece(test_logits,test_y,1.0):.3f}   after {ece(test_logits,test_y,T):.3f}")

In [ ]:
# ── Cell 5 · ≥99%-precision threshold on VALIDATION (known classes) ──
# Open-set selective prediction: we "answer" only when the prediction is one of the FIVE
# known models AND calibrated confidence >= threshold. Predicting 'others' = reject (route to
# a human); below threshold = abstain. Precision is measured on the answers that matter.
TARGET = 0.99
OTH = CLASSES.index('others') if 'others' in CLASSES else -1     # -1 => no reject class (5-class back-compat)

def conf_pred(logits, T):
    p = torch.softmax(logits/T, 1); c, pr = p.max(1); return c.numpy(), pr.numpy()

vc, vp = conf_pred(val_logits, T); v_true = val_y.numpy()
v_known = vp != OTH                                    # predicted a known model (not 'others')
best_t, best_cov = None, -1.0
for t in np.unique(vc):
    m = v_known & (vc >= t)
    if m.sum() >= 20 and (vp[m] == v_true[m]).mean() >= TARGET and m.mean() > best_cov:
        best_cov, best_t = m.mean(), float(t)

if best_t is None:
    cand = [(m.mean(), (vp[m]==v_true[m]).mean(), float(t))
            for t in np.unique(vc) for m in [v_known & (vc>=t)] if m.sum() >= 20]
    cov, pr, best_t = max(cand, key=lambda z: z[1])
    print(f"⚠️ {TARGET:.0%} not reachable on val. Best known-precision {pr:.1%} @ {cov:.1%} coverage (thr={best_t:.3f}).")
else:
    mm = v_known & (vc >= best_t)
    print(f"threshold for ≥{TARGET:.0%} KNOWN-class precision (chosen on VAL): {best_t:.3f}")
    print(f"  → val known-precision {(vp[mm]==v_true[mm]).mean():.1%}, val coverage {mm.mean():.1%}")

In [ ]:
# ── Cell 6 · Apply the val-chosen threshold ONCE to the sealed test ──
tc, tp = conf_pred(test_logits, T); t_true = test_y.numpy()
t_known = tp != OTH
answered = t_known & (tc >= best_t)                    # trusted known-model answers
print(f"SEALED TEST @ threshold {best_t:.3f} (open-set, precision on the 5 known models):")
print(f"  known-precision on answered : {(tp[answered]==t_true[answered]).mean():.1%}")
print(f"  coverage (trusted answers)  : {answered.mean():.1%}   of all test images")
print(f"  routed to 'others' (reject) : {(tp==OTH).mean():.1%}")
print(f"  abstained (low confidence)  : {(t_known & (tc < best_t)).mean():.1%}")
ref = t_known
print(f"  (reference: no abstention -> known-precision {(tp[ref]==t_true[ref]).mean():.1%} at {ref.mean():.1%} coverage)")

# precision–coverage (risk–coverage) curve on test, restricted to known-model answers
cov, prec = [], []
for t in np.linspace(tc.min(), tc.max(), 120):
    mm = t_known & (tc >= t)
    if mm.sum() >= 10: cov.append(mm.mean()); prec.append((tp[mm]==t_true[mm]).mean())
plt.figure(figsize=(6,4))
plt.plot(cov, prec, '-')
plt.axhline(TARGET, color='r', ls='--', label=f'{TARGET:.0%} precision')
plt.scatter([answered.mean()], [(tp[answered]==t_true[answered]).mean()], color='k', zorder=5, label='operating point')
plt.xlabel('coverage (fraction given a known-model answer)'); plt.ylabel('precision on known-model answers')
plt.title('Precision–coverage — sealed test (5 known models)'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 7 · Reliability diagrams (before vs after calibration) ─
def reliability(logits, y, T, ax, title):
    p = torch.softmax(logits/T, 1); conf, pred = p.max(1)
    conf, pred, y = conf.numpy(), pred.numpy(), y.numpy()
    edges = np.linspace(0, 1, 11); xs, ys = [], []
    for i in range(10):
        mm = (conf > edges[i]) & (conf <= edges[i+1])
        if mm.sum(): xs.append(conf[mm].mean()); ys.append((pred[mm]==y[mm]).mean())
    ax.plot([0,1],[0,1],'k--',alpha=.5); ax.plot(xs, ys, 'o-')
    ax.set_title(title); ax.set_xlabel('confidence'); ax.set_ylabel('accuracy')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
fig, axs = plt.subplots(1, 2, figsize=(9,4))
reliability(test_logits, test_y, 1.0, axs[0], 'before (T=1)')
reliability(test_logits, test_y, T,   axs[1], f'after (T={T:.2f})')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 8 · The most confident precision-breaking mistakes ─────
# Confident KNOWN-class predictions that are wrong — these are exactly what abstention must
# catch (e.g. an 'others' car confidently called a known model, or a look-alike sedan mixup).
bad = np.where((tp != t_true) & t_known)[0]
bad = bad[np.argsort(-tc[bad])][:12]
fig, ax = plt.subplots(2, 6, figsize=(15, 5.5))
for a, idx in zip(ax.ravel(), bad):
    path, _ = test_ds.samples[idx]
    a.imshow(Image.open(path)); a.axis('off')
    a.set_title(f"true {CLASSES[int(test_y[idx])]}\npred {CLASSES[tp[idx]]} ({tc[idx]:.2f})", fontsize=8)
for a in ax.ravel()[len(bad):]: a.axis('off')
plt.suptitle("Most confident wrong KNOWN-model predictions — what the abstention threshold removes")
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 9 · Save calibration to the artifact + back up ─────────
cfg['temperature'] = float(T)
cfg['abstain_threshold'] = float(best_t)
cfg['precision_target'] = TARGET
cfg['precision_scope'] = 'five_known_models'
json.dump(cfg, open('/content/artifacts/config.json','w'), indent=2)
OUT_ZIP = Path(ART_ZIP).name          # re-save under the same artifacts name it was loaded from
!cd /content && zip -q -r "$OUT_ZIP" artifacts >/dev/null
!cp /content/"$OUT_ZIP" /content/drive/MyDrive/CapstoneCars/
print(f"✅ temperature + abstain threshold saved to config.json and backed up as {OUT_ZIP}")
print("   config now:", {k: cfg[k] for k in ['winner','head','temperature','abstain_threshold','precision_scope']})